In [2]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

In [3]:
def retrieve_documents(dataset_id, api_key, query, search_method="semantic_search", weights=0.5,top_k=3):
    """
    从数据集中检索文档。

    参数:
    - dataset_id (str): 数据集 ID。
    - api_key (str): API 密钥。
    - query (str): 查询关键词。
    - search_method (str): 检索方法，默认为 "keyword_search"。
    - top_k (int): 返回的结果数量，默认为 1。

    返回:
    - dict: API 的响应结果。
    """
    # 设置请求 URL 和头部
    url = f"http://172.16.2.77:56966/v1/datasets/{dataset_id}/retrieve"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 定义请求体
    payload = {
        "query": query,
        "retrieval_model": {
            "search_method": search_method,
            "reranking_enable": False,
            "reranking_mode": None,
            "reranking_model": {
                "reranking_provider_name": "",
                "reranking_model_name": ""
            },
            "weights": weights,
            "top_k": top_k,
            "score_threshold_enabled": False,
            "score_threshold": None
        }
    }

    # 发送 POST 请求
    response = requests.post(url, headers=headers, data=json.dumps(payload))

    # 返回解析后的响应
    if response.status_code == 200:
        return response.json()
    else:
        print({"error": response.status_code, "message": response.text})
        return {"error": response.status_code, "message": response.text}


In [4]:
# 使用示例
dataset_id = "c02c5910-993a-4bed-8985-c7f23572b837"
api_key = "dataset-F8zSYohCngBjVNsh2NGBv5Bl"  # 替换为实际 API 密钥
query = "甲状腺炎"

result = retrieve_documents(dataset_id, api_key, query)
print(result)
print("实际返回结果量：",len(result["records"]))

for rel in result["records"]:
    print(rel["segment"]["document"])
    print(rel["segment"]["content"])
    print(len(rel["segment"]["content"]))

    print(rel["score"])

{'query': {'content': '甲状腺炎'}, 'records': [{'segment': {'id': 'e36e6467-9cac-4fc9-b8eb-831b33f010a9', 'position': 446, 'document_id': '7f7274db-1fbb-4415-8310-0d9bec5f70bf', 'content': '{"disease_n": "甲状腺炎", "disease_keyword": ["甲状腺炎", "急性甲状腺炎", "局灶性亚甲炎", "桥本氏甲状腺炎", "慢性自身免疫性甲状腺炎", "桥本氏病", "桥本氏炎", "淋巴细胞甲状腺炎", "亚甲炎", "甲炎"]}', 'answer': None, 'word_count': 137, 'tokens': 180, 'keywords': ['disease', '局灶性', '氏病', '免疫性', '氏炎', 'keyword', '甲状腺炎', '亚甲炎', '甲炎', '桥本'], 'index_node_id': 'd9be84d5-77c7-45f3-a182-71b2ec889791', 'index_node_hash': '3223235d0997c4016c1ad4e04b77cb8b755e060c568aa78afde89ad0dcd62bd7', 'hit_count': 0, 'enabled': True, 'disabled_at': None, 'disabled_by': None, 'status': 'completed', 'created_by': '71416823-659b-4a69-a747-9b2fe3670645', 'created_at': 1733127335, 'indexing_at': 1733127337, 'completed_at': 1733127347, 'error': None, 'stopped_at': None, 'document': {'id': '7f7274db-1fbb-4415-8310-0d9bec5f70bf', 'data_source_type': 'upload_file', 'name': '中再疾病函数对照表', 'doc_typ

# 大模型支持

In [5]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

In [6]:
user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input}]
qa_base(input)

'中国的首都是北京。'

# 核保知识重建

In [7]:
df_dise_2_keyword = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)
df_dise_2_conc = pd.read_csv("./utils/disease_2_conclusion_v4.csv",keep_default_na=False)

In [8]:
df_dise_2_conc[df_dise_2_conc["disease_name"]=="脂肪肝*"].head()

,disease_name,file_name,conclusion
733,脂肪肝*,肝脏.txt,"{'疾病': '脂肪肝*', '资料': '病史，肝功能、腹部超声、肝脏瞬时弹性波扫描、肝脏..."
734,脂肪肝*,肝脏.txt,"{'疾病': '脂肪肝*', '资料': '病史，肝功能、腹部超声、肝脏瞬时弹性波扫描、肝脏..."
735,脂肪肝*,肝脏.txt,"{'疾病': '脂肪肝*', '资料': '病史，肝功能、腹部超声、肝脏瞬时弹性波扫描、肝脏..."
736,脂肪肝*,肝脏.txt,"{'疾病': '脂肪肝*', '资料': '病史，肝功能、腹部超声、肝脏瞬时弹性波扫描、肝脏..."
737,脂肪肝*,肝脏.txt,"{'疾病': '脂肪肝*', '资料': '如有重度脂肪肝、酒精性脂肪肝，需综合体格（BMI..."


In [9]:
#构建疾病及其相关关键词map, df来自中再疾病函数对照表
disease_n_2_list = {}
for idx,row in df_dise_2_keyword.iterrows():
    disease_n = row["disease_n"].strip()
    disease_list = eval(row["disease_list"])
    # print(type(disease_list))
    try:
        if disease_n  not in disease_n_2_list and disease_n!="":
            disease_n_2_list[disease_n] = []
        disease_n_2_list[disease_n].extend(disease_list)
    except Exception as es:
        print(es)

''
''
''
''
''


In [31]:
def clear_text(text):
    text = text.strip()
    text = text.replace("\n","")
    # text = text.replace("*","")
    text = text.replace("\t","")
    text = text.replace("2","II")
    text = text.replace("1","I")
    return text

disease_ns = disease_n_2_list.keys()
disease_names = list(set(df_dise_2_conc["disease_name"].tolist()))


disease_ns = [clear_text(k) for k in disease_ns]
disease_names = [clear_text(k) for k in disease_names]

print("disease_ns",len(disease_ns))
print("disease_names",len(disease_names))

intersection = set(disease_ns) & set(disease_names)
print(len(intersection))
disease_names_only = set(disease_names) - set(disease_ns)
print(f"disease_name only have:{len(disease_names_only)}")
disease_ns_only = set(disease_ns) - set(disease_names)
print("disease_ns only have:",len(disease_ns_only))

disease_map = {
    "胆囊疾病":"胆结石",
    "肺结节病":"肺结节",
    "脂肪肝":"脂肪肝*",
    "乳腺结节":"乳腺结节、囊肿、占位、异常回声",
    "肾结石":"泌尿系结石（无高血压和肾功能损害）",
    "心率不齐":"心率失常",
    "肺动脉瓣关闭不全":"肺动脉瓣疾病"
}

disease_ns 768
disease_names 227
115
disease_name only have:112
disease_ns only have: 653


# 对测试用例进行RAG生成

In [35]:
# df = pd.read_csv("./规则引擎对比结果-评点数据.csv",keep_default_na=False)
df = pd.read_csv("./result_recall_disease/核保结论_规则引擎_vs_RAG_disease_recall.csv",keep_default_na=False)

In [36]:
df.columns

Index(['姓名', 'disease_name', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果', 'RAG核保结论', 'recall_query',
       'recall_disease', 'recall_conclsion'],
      dtype='object')

In [24]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_conclusions = []
recall_time_use = []
for idx,row in df.iterrows():
    
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    print("病人基本信息",basic_info)
    print(image_report)
    # if image_report != "":
    #     image_report = json.loads(image_report)
    #     print(type(image_report))
    #     query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    # else:
    #     query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    query = diagnose
    
    s_time = time.time()
    try:
        recall_kb = retrieve_documents(dataset_id, api_key, query,search_method="semantic_search",top_k=5)
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)

        # print("核保结论实际召回量：",len(recall_kb["records"]))
        disease_list = [eval(rel["segment"]["content"]) for rel in recall_kb["records"]]
        first_disease = disease_list[0]["disease_n"]
        print(first_disease)
        if first_disease in intersection:
             first_disease = first_disease
        else:
            first_disease = disease_map.get(first_disease,"")
        ans_list = df_dise_2_conc[df_dise_2_conc["disease_name"]==first_disease]["conclusion"].tolist()
        # print("召回的核保结论：",ans_list)
    except Exception as es:
        print(es)
        ans_list = []
    

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
                3、诺核保结论为空则不返回最终核保结论
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,output_struct=output_struct)
    # print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)

    results_underwriting.append(result)
    recall_query.append(disease_list)
    recall_conclusions.append(ans_list)

    

病人基本信息 年龄:34,性别:女性,临床诊断:宫腔内稍高回声团,考虑子宫内膜息肉,影像报告:{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
recall time use: 0.13078713417053223
子宫内膜息肉
病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54

In [52]:
sum(recall_time_use)/len(recall_time_use)

0.47619374841451645

# 对RAG结果进行check

In [53]:
print(len(results_underwriting))
print(len(recall_query))

32
32


In [27]:
# df["RAG核保结论"] = results_underwriting
df["recall_disease"] = recall_query
df["recall_conclsion"] = recall_conclusions

In [29]:
df.to_csv("./result_recall_disease/核保结论_规则引擎_vs_RAG_disease_recall.csv",index=False)

# Dify大模型本地部署
- ChatGLM,可以部署LLM
- OpenLLM,可以部署LLM,embedding
